In [ ]:
"""Test khả năng dịch từ tiếng việt sang tiếng anh và ngược lại"""

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-8B"

# Load tokenizer and model
assert torch.cuda.is_available(), "CUDA PyTorch is required; restart the Jupyter kernel."
print(f"Using GPU: {torch.cuda.get_device_name(0)}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
    max_memory={0: "3GiB", "cpu": "10GiB"},
    offload_folder="qwen8b_offload",
    offload_state_dict=True
)

# Change this prompt to test different capabilities
prompt = """
Complete these tasks:

1. Translate this Vietnamese sentence into English:
   "Bệnh nhân bị tăng huyết áp và đang sử dụng amlodipine."

2. Extract diseases, symptoms, and medications as valid JSON.

3. Write a Python function that checks whether a number is prime.

Keep your answer concise.
"""

messages = [
    {
        "role": "system",
        "content": "You are a precise and helpful AI assistant."
    },
    {
        "role": "user",
        "content": prompt
    }
]

# Convert chat messages into Qwen's required prompt format
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False  # Change to True to test reasoning mode
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.get_input_embeddings().weight.device)

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20
    )

# Remove the original input tokens and keep only Qwen's response
response_ids = generated_ids[0, inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    response_ids,
    skip_special_tokens=True
)

print("\n========== QWEN RESPONSE ==========\n")
print(response)